In [ ]:
from torch import nn
from torch.distributions import Categorical

class Policy(nn.Module):
    def __init__(self, obs_dim, n_actions, hidden=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(obs_dim, hidden), nn.Tanh(),
            nn.Linear(hidden, hidden), nn.Tanh(),
            nn.Linear(hidden, n_actions)
        )

    def forward(self, obs):
        logits = self.net(obs)
        return Categorical(logits=logits)


class Value(nn.Module):
    def __init__(self, obs_dim, hidden=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(obs_dim, hidden), nn.Tanh(),
            nn.Linear(hidden, hidden), nn.Tanh(),
            nn.Linear(hidden, 1)
        )
    
    def forward(self, obs):
        return self.net(obs).squeeze(-1)

In [ ]:
import torch
from torch.optim import Adam


class PPO:
    def __init__(self, policy, value, env, pi_lr, vf_lr, eps, k, num_steps, num_epochs, minibatch_size, gamma):
        self.policy = policy
        self.value = value
        self.pi_optimizer = Adam(policy.parameters(), lr=pi_lr)
        self.vf_optimizer = Adam(value.parameters(), lr=vf_lr)
        self.env = env
        self.eps = eps
        self.k = k
        self.num_steps = num_steps
        self.num_epochs = num_epochs
        self.minibatch_size = minibatch_size
        self.gamma = gamma

    @torch.no_grad()
    def collect_rollouts(self, num_steps):
        obs_list, action_list, reward_list, terminated_list, truncated_list, logp_list, value_list = [], [], [], [], [], [], []
        obs, _ = self.env.reset(seed=42)
        for _ in range(num_steps):
            obs_t = torch.as_tensor(obs, dtype=torch.float32)
            dist = self.policy(obs_t)
            action = dist.sample()
            logp = dist.log_prob(action)
            value = self.value(obs_t)
            next_obs, reward, terminated, truncated, _ = self.env.step(action.item())
            obs_list.append(obs_t)
            action_list.append(action)
            reward_list.append(reward)
            terminated_list.append(terminated)
            truncated_list.append(truncated)
            logp_list.append(logp)
            value_list.append(value)

            obs = next_obs
            if terminated or truncated:
                obs, _ = self.env.reset()
        
        last_value = self.value(torch.as_tensor(obs, dtype=torch.float32))

        return {
            "obs": torch.stack(obs_list),
            "act": torch.stack(action_list),
            "reward": torch.tensor(reward_list, dtype=torch.float32),
            "terminated": torch.tensor(terminated_list, dtype=torch.float32),
            "logp_old": torch.stack(logp_list),
            "value": torch.stack(value_list),
            "last_value": last_value,
        }
    
    def compute_advantages(self, reward, value, terminated, last_value):
        T = len(reward)
        next_values = torch.empty(T)
        next_values[:-1] = value[1:]
        next_values[-1] = last_value
        next_nonterminal = 1.0 - terminated
        advantages = reward + self.gamma * next_values * next_nonterminal - value
        returns = advantages + value
        return advantages, returns

    def train(self):
        for _ in range(self.k):
            batch = self.collect_rollouts(self.num_steps)
            obs, act, logp_old = batch["obs"], batch["act"], batch["logp_old"]

            with torch.no_grad():
                advantages, returns = self.compute_advantages(batch["reward"], batch["value"], batch["terminated"], batch["last_value"])
                advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-8)

            for _ in range(self.num_epochs):
                idx = torch.randperm(self.num_steps)
                for start in range(0, self.num_steps, self.minibatch_size):
                    mb = idx[start:start+self.minibatch_size]
                    logp_new = self.policy(obs[mb]).log_prob(act[mb])
                    ratio = torch.exp(logp_new - logp_old[mb])
                    adv = advantages[mb]
                    g = torch.where(adv >= 0, (1 + self.eps) * adv, (1 - self.eps) * adv)
                    loss = -torch.min(ratio * adv, g).mean()
                    self.pi_optimizer.zero_grad()
                    loss.backward()
                    self.pi_optimizer.step()

                    value_loss = ((self.value(obs[mb]) - returns[mb])**2).mean()
                    self.vf_optimizer.zero_grad()
                    value_loss.backward()
                    self.vf_optimizer.step()

In [ ]:
@torch.no_grad()
def compute_rewards(env, policy):
    policy.eval()
    observation, _ = env.reset(seed=42)
    num_episodes = 1
    rewards = 0.0
    for _ in range(1000):
        action = policy(torch.as_tensor(observation, dtype=torch.float32)).sample()
        observation, reward, terminated, truncated, _ = env.step(action.item())
        rewards += reward
        if terminated or truncated:
            num_episodes += 1
            observation, _ = env.reset(seed=42)
    return rewards / num_episodes

In [ ]:
import gymnasium as gym

env = gym.make("CartPole-v1")

policy = Policy(env.observation_space.shape[0], env.action_space.n)
value = Value(env.observation_space.shape[0])

print(compute_rewards(env, policy))

policy.train()
PPO(policy, value, env, 1e-3, 1e-3, 0.2, 100, 2048, 10, 64, 0.99).train()

print(compute_rewards(env, policy))

env.close()